In [29]:
import networkx as nx
import pandas as pd
import torch
import pickle
from collections import defaultdict
import re
from tqdm import tqdm

## 1. extract protein embedddings

In [2]:
def load_graph(filepath):
    with open(filepath, 'rb') as f:
        G = pickle.load(f)
    print(f"Load graph with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")
    return G

In [98]:
def get_protein_embeddings(graph_path, embeddings_path):
    
    G = load_graph(graph_path)
    protein_nodes = []
    for node, attr in G.nodes(data=True):
        # print(node)
        # print(attr)

        if attr.get('label')=='Protein':
            protein_nodes.append(node)
    
    df_embedding = pd.read_csv(embeddings_path, index_col=0)
    embeddings = df_embedding[df_embedding.index.isin(protein_nodes)]
    return embeddings

In [80]:
graph_path = "../../EdgeAssignmentTask/neighborhood_selection/results/pykeen_kge/AD_KG/G_merge_AD_KG_healthy.pkl"
embedding_path = "../../EdgeAssignmentTask/neighborhood_selection/results/pykeen_kge/AD_KG/embedding.csv"

embeddings = get_protein_embeddings(graph_path, embeddings_path=embedding_path)
embeddings

Load graph with 7820 nodes and 24461 edges


,Component_1,Component_2,Component_3,Component_4,Component_5,Component_6,Component_7,Component_8,Component_9,Component_10,...,Component_247,Component_248,Component_249,Component_250,Component_251,Component_252,Component_253,Component_254,Component_255,Component_256
"fus(p(UniProtKB:""LRP4_MOUSE"",""p.1_1747""),p(UniProtKB:""CD4_MOUSE"",""p.422_457""))",1.528266,0.241949,0.284356,0.857318,0.165116,0.762016,0.380046,0.386629,0.339537,2.083950,...,0.438657,-2.984491,-0.831968,-0.539908,0.588438,0.472758,1.726272,-0.789979,2.853453,0.231803
"p(CHEBI:""amyloid-beta polypeptide 42"")",-0.838868,-0.894812,-1.665828,-0.544204,-3.117030,-2.072838,-0.127108,0.739820,1.961052,-2.418546,...,0.842275,-0.236417,-0.191154,-1.368578,-0.375749,0.474667,0.663514,-0.414643,-0.038772,-2.395431
"p(CHEBI:""amyloid-beta"")",0.296918,0.303208,0.231687,0.017844,0.004659,0.132376,-0.039097,0.163802,-0.232891,0.196605,...,0.578156,0.033174,0.184513,0.191104,0.201498,0.162784,-0.037819,0.310565,-0.474039,0.208725
"p(CHEBI:""cholesterol"")",-1.612742,-1.458753,2.437611,-0.055724,-1.314280,0.290219,0.310820,0.897895,0.628886,0.065758,...,-1.217489,-0.778933,-0.942460,-0.172261,-0.661084,-1.984987,-0.728508,0.095277,1.586002,-0.674741
"p(CONSO:""APP C-terminally truncated carboxyl-terminal fragments"")",0.010255,0.306495,-3.339482,-0.005943,-0.400550,-0.502296,-1.220509,0.500731,0.047272,1.457177,...,-0.134347,1.284306,0.072084,-0.284287,0.555389,0.390514,-1.781293,1.313754,-0.156425,0.027209
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"p(UniProtKB:""VLDLR_RAT"")",-0.238215,0.217475,-1.015950,-4.427256,0.293364,-0.042365,-0.475032,-0.266107,0.393299,1.241113,...,0.573863,1.430837,-0.259398,1.168401,-2.742837,1.496125,0.770096,-1.276596,-0.742884,1.000416
"p(UniProtKB:""VLDLR_RAT"",var(""O-linked glycosylation domain_del""))",-0.439215,-1.923677,1.165449,-0.110652,1.207234,-0.202278,2.042466,-0.775431,0.611722,-1.361381,...,-0.086732,0.599778,0.689020,0.902289,0.840229,0.755602,-0.089410,0.982870,0.314392,0.261414
"p(UniProtKB:""VPS35_HUMAN"")",-0.009121,0.515759,-0.728741,-0.062857,-0.058965,0.455048,-0.412247,0.849345,0.153839,-0.200507,...,0.371100,0.922370,-0.060566,-0.226681,-0.437978,-0.003002,-0.485556,0.295658,-0.718291,-0.033799
"p(UniProtKB:""VTDB_MOUSE"")",0.210360,-0.446954,0.293490,-1.524487,-1.331538,0.407359,0.320301,-0.081390,-0.842339,-1.081459,...,-1.134385,0.121495,-1.411129,-1.432221,0.475971,-1.593193,0.561471,-0.214770,-0.365978,0.175966


## 2. create patient-protein network
+ patient features: gene expression
+ proteins features: embeddings
+ edge_weight: normalized expression value
+ edge_type: with/without specifying disease/control

#### Generate sample scoring

In [10]:
import sys
import os

# Get the path to the 'root' directory
root_path = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

# Add root to the system path
if root_path not in sys.path:
    sys.path.append(root_path)

# Now can import function from upper-folder
from data_processing.sample_scoring import do_radical_search

In [30]:
# normalize expression data
def minmax_normalize_exp(exp_df):
    if exp_df.shape[0] > exp_df.shape[1]:
        exp_df = exp_df.transpose()
    
    # drop genes with no variation
    exp_df = exp_df.loc[:, exp_df.std() > 0]
    # Using median is usually safer for gene expression
    exp_df = exp_df.fillna(exp_df.median())
    # normalize safely
    min_val = exp_df.min()
    max_val = exp_df.max()
    exp_norm = (exp_df - min_val) / (max_val - min_val + 1e-9)
    # final fill-na
    exp_norm = exp_norm.fillna(0)
    return exp_norm, exp_df

In [ ]:
class PatientProteinNetworkGenerator():
    def __init__(self):
        self.pattern_hgnc =  r'^p\(HGNC:"([^"]+)"\)$'
        self.pattern_uniprotkb =  r'^p\(UniProtKB:"([^"_%]+)_[A-Z]+"\)$'
        self.pattern_dash = r'(\w+)_HUMAN'
    
    def gene_symbol_extractor(self, text, pattern:str):
        # ^ ensures start at the beginning, $ ensures end at the ')'
        match = re.search(pattern, text)
        if match:
            target = match.group(1)
            return target.upper()
        return None

    def get_symbol_mapping(self, nodes:list):
        """
        Universal helper function to create {gene_symbol: [kg_nodes]} mapping 
        across ad_kg, control_kg, ppi_kg, and prime_kg.
        """
        mapping = defaultdict(list)
        
        for node in nodes:
            # Ensure node is a string and skip empty/complex modifications
            if not isinstance(node, str) or any(x in node for x in ['frag(', 'var(', 'pmod(', 'loc(', ]):
                continue
                
            symbol = None
            
            # Case 1: BEL Namespace syntax (ad_kg & control_kg) -> p(HGNC:"...") or p(UniProtKB:"...")
            if node.startswith('p(') and node.endswith(')'):
                if 'HGNC' in node:
                    symbol = self.gene_symbol_extractor(node, self.pattern_hgnc)
                elif 'UniProtKB' in node:
                    symbol = self.gene_symbol_extractor(node,self.pattern_uniprotkb)
                else:
                    # Fallback for unexpected namespaces inside p(...) like p(FIXME:"...")
                    match = re.search(r'["\']([^"\']+)["\']', node)
                    if match:
                        symbol = match.group(1)
                    #continue

            # Case 2: handle (ppi_kg) -> AL1A1_HUMAN
            elif '_' in node and any(suffix in node for suffix in ['_HUMAN']):
                if 'UniProtKB' not in node and 'HGNC' not in node:
                    symbol = self.gene_symbol_extractor(node, self.pattern_dash)
            else:
                # Case 3: handle Gene/Protein symbols in PrimeKg
                clean_node = node.strip().upper()
                symbol=clean_node

            # Append to our list mapping if a valid symbol was resolved
            if symbol:
                symbol_upper = symbol.upper()
                mapping[symbol_upper].append(node)
                
        return dict(mapping)
    
    def generate(self, KG, 
                 data:pd.DataFrame, 
                 embeddings:pd.DataFrame, 
                 exp:pd.DataFrame, 
                 include_isolated_nodes=False,
                 label_relation=False):

        patient_labels = data['label'].to_dict()
        scores = data.drop(columns=['label'])

        if include_isolated_nodes:
            protein_nodes = [node for node, attr in KG.nodes(data=True) if 'Protein' in attr.get('label')]
        else:
            protein_nodes = embeddings.index.tolist()

        symbol_to_kg_node = self.get_symbol_mapping(list(protein_nodes))
        common_proteins = [s for s in scores.columns if s in symbol_to_kg_node]
        print(f"There are {len(common_proteins)} common Gene Symbols in GE and AD_Healthy_KGs")

        # filter out gene-symbol columns not in KGs, stack them vertically to a series
        sparse_data: pd.Series = scores[common_proteins].stack() # type: ignore
        # filter out gene-symbol columns with ecdf_score=0 (not extremely expressed)
        mask = sparse_data != 0
        radicals = sparse_data.loc[mask]

        summary_df = pd.DataFrame(0, index=data.index, columns=['pos_edges', 'neg_edges'])
        summary_df['pos_linked_nodes'] = [[] for _ in range(len(summary_df))] # Initialize with empty lists
        summary_df['neg_linked_nodes'] = [[] for _ in range(len(summary_df))]
        summary_df['label'] = data['label'].to_list()

        # Add patient nodes to Network
        exp_norm, exp_df = minmax_normalize_exp(exp)

        G = nx.MultiDiGraph()
        relation_map = {1: f'up_reg', -1: f'down_reg'}
        h_relation_map = {1: f'up_reg_h', -1: f'down_reg_h'}
        d_relation_map = {1: f'up_reg_d', -1: f'down_reg_d'}
        # Add protein nodes and edges
        for key, val in tqdm(radicals.items(), total=len(radicals), desc="Linking Samples"):
            if not (isinstance(key, tuple) and len(key) == 2):
            # unexpected index shape — skip or handle explicitly
                continue
            patient, symbol = key

            for protein_node in symbol_to_kg_node[symbol]:
                if label_relation:
                    if 'UniProtKB' in protein_node:
                        rel = h_relation_map.get(int(val))
                    else:
                        rel=d_relation_map.get(int(val))
                else:
                    rel=relation_map.get(int(val))
                
                # add patient node to G
                if not G.has_node(patient):
                    G.add_node(patient, 
                            x=exp_df.loc[patient].values, 
                            y=int(patient_labels[patient]),
                            label='Patient',
                                type='Patient')
                # add protein nodes to G
                if not G.has_node(protein_node):
                    protein_y = 0 if 'UniProtKB' in protein_node else 1 
                    #print(protein_y)
                    try:
                        protein_x = embeddings.loc[protein_node].values
                    except:
                        protein_x = None
                    G.add_node(protein_node,
                            x=protein_x,
                            y=protein_y,
                            label='Protein',
                            type='Protein')
                
                # edge weight: normalized expression value
                raw_weight = exp_norm.at[patient, symbol]
                weight_value = float(raw_weight) # type: ignore
                
                # Use relation as the 'key' for MultiDiGraph edges
                G.add_edge(patient, protein_node, key=rel, relation=rel, weight=weight_value)
                G.add_edge(protein_node, patient, key=f'rev_{rel}', relation=f'rev_{rel}',weight=weight_value)
                
                col = 'pos_edges' if int(val) == 1 else 'neg_edges'
                summary_df.at[patient, col] += 1 # type: ignore
                linked_nodes_col = 'pos_linked_nodes' if int(val)==1 else 'neg_linked_nodes'
                summary_df.at[patient,linked_nodes_col].append(protein_node) # type: ignore
        
        return G, summary_df

In [ ]:
# 2. load exp, design data
exp_df = pd.read_csv("../../data/ADNI/cleaned_gene_expression_data.csv",
                                index_col=0)
design = pd.read_csv("../../data/ADNI/design_with_real_target.tsv",
                     sep='\t',
                     index_col=0)
design['Target'] = design['Old_Target'].map({'Control':0, 'Disease':1})

# 3. sample scoring
data, summary = do_radical_search(data=exp_df,
                                  design=design,
                                  threshold=5,
                                  control=0,
                                  control_based=True)

In [119]:
# script

# 1. prepare protein embeddings
graph_path = "../../EdgeAssignmentTask/neighborhood_selection/results/pykeen_kge/AD_KG/G_merge_AD_KG_healthy.pkl"
kg = load_graph(graph_path)
embedding_path = "../../EdgeAssignmentTask/neighborhood_selection/results/pykeen_kge/AD_KG/embedding.csv"
embeddings = get_protein_embeddings(graph_path, embeddings_path=embedding_path)


# 4. generate network
ppng = PatientProteinNetworkGenerator()
G, summary_df = ppng.generate(KG=kg,data=data, embeddings=embeddings, exp=exp_df)

Load graph with 7820 nodes and 24461 edges
Load graph with 7820 nodes and 24461 edges
There are 865 common Gene Symbols in GE and AD_Healthy_KGs


Linking Samples: 100%|██████████| 64934/64934 [00:38<00:00, 1677.28it/s]


In [117]:
summary_df

,pos_edges,neg_edges,pos_linked_nodes,neg_linked_nodes,label
116_S_1249,26,46,"[p(HGNC:""ABCA1""), p(HGNC:""AGTR1""), p(HGNC:""CAL...","[p(HGNC:""ACE""), p(HGNC:""APOA2""), p(HGNC:""APOB""...",0
037_S_4410,81,150,"[p(HGNC:""ADD3""), p(HGNC:""AKT3""), p(HGNC:""ANXA1...","[p(HGNC:""ABCA2""), p(HGNC:""ABCA7""), p(HGNC:""ABC...",0
006_S_4153,42,22,"[p(HGNC:""AGTR1""), p(HGNC:""APBB2""), p(HGNC:""BAC...","[p(HGNC:""ANXA1""), p(HGNC:""CACNG7""), p(HGNC:""CT...",1
116_S_1232,34,9,"[p(HGNC:""ADIPOQ""), p(HGNC:""ALB""), p(HGNC:""AR"")...","[p(HGNC:""CDK5R1""), p(HGNC:""CTSG""), p(HGNC:""CXC...",0
099_S_4205,22,53,"[p(HGNC:""APC""), p(HGNC:""BLMH""), p(HGNC:""CALCR""...","[p(HGNC:""ABCG2""), p(HGNC:""AGO2""), p(HGNC:""APOA...",1
...,...,...,...,...,...
009_S_2381,60,16,"[p(HGNC:""ABCC8""), p(HGNC:""ALB""), p(HGNC:""APOA2...","[p(HGNC:""BCHE""), p(HGNC:""CD274""), p(HGNC:""CD40...",1
053_S_4557,24,16,"[p(HGNC:""ACE""), p(HGNC:""AGTR1""), p(HGNC:""CAPN1...","[p(HGNC:""APBB2""), p(HGNC:""APC""), p(HGNC:""CDH2""...",1
073_S_4300,23,26,"[p(HGNC:""ANGPT2""), p(MGI:""Angpt2""), p(HGNC:""AT...","[p(HGNC:""ADAM17""), p(HGNC:""ARMS2""), p(HGNC:""CA...",1
041_S_4014,119,64,"[p(HGNC:""ABCA7""), p(HGNC:""ABCC9""), p(HGNC:""ADI...","[p(HGNC:""ADD3""), p(HGNC:""APP""), p(MGI:""App""), ...",0


### Caveats
+ the BaseKGs have isolated nodes --> no triples after converting nx.graph to pd EdgeList --> no embeddings for those nodes(include proteins) generated
+ I add all protein nodes to the SampleProteinNetwork, but those without embeddings, I give x=None;
+ 

In [108]:
edgelist = pd.read_csv("../../EdgeAssignmentTask/neighborhood_selection/results/pykeen_kge/AD_KG/Edgelist_merge_['PPI_KG', 'Prime_KG', 'AD_KG']_healthy.csv", index_col=0)
train = pd.read_csv(
    "../../EdgeAssignmentTask/neighborhood_selection/results/pykeen_kge/AD_KG/train.edgelist",
    sep=r'\t',
    header=None,
    names=['source', 'relation','target'],
    engine='python'
)

all_nodes = set(edgelist['source']) | set(edgelist['target'])
train_nodes = set(train['source']) | set(train['target'])
missing = len(all_nodes) - len(train_nodes)
missing  # should be ~1287

586

In [109]:
len(all_nodes)

7119

In [112]:
graph_path = "../../EdgeAssignmentTask/neighborhood_selection/results/pykeen_kge/AD_KG/G_merge_AD_KG_healthy.pkl"
ad_h_kg = load_graph(graph_path)

len(list(nx.isolates(ad_h_kg)))

Load graph with 7820 nodes and 24461 edges


701

In [114]:
len(list(nx.isolates(ad_kg)))

705